# AI-driven Water Quality Monitoring: Person C (App & Deployment)

**Role:** App & Deployment Engineer

**Deliverables:**
- Interactive Web Dashboard (`personC_app/app.py`)
- Production Dockerfile (`personC_app/Dockerfile`)
- Batch Inference & Testing dataset (`personC_app/sample_water_test.csv`)
- End-to-end deployment documentation

---

## 1. Setup & Environment Verification

In [1]:
import os, sys, json, time, pathlib
import numpy as np, pandas as pd
import joblib, shap
import warnings
warnings.filterwarnings('ignore')

MODEL_PATH = 'kaggle_output_personB/models/best_model.joblib'
META_PATH = 'kaggle_output_personB/models/model_meta.json'
OUTPUTS_DIR = 'kaggle_output_personB/outputs_B'

print('Model file exists:', os.path.exists(MODEL_PATH))
print('Metadata file exists:', os.path.exists(META_PATH))

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model file exists: True
Metadata file exists: True


## 2. Load Model & Metadata

In [2]:
model = joblib.load(MODEL_PATH)
with open(META_PATH, 'r') as f:
    meta = json.load(f)

print('Loaded Model:', type(model).__name__)
print('Model Name:', meta['model_name'])
print('Optimized Decision Threshold:', meta['threshold'])
print('Total Features:', len(meta['features']))
print('Test ROC-AUC:', meta['test_metrics']['roc_auc'])
print('Test Recall:', meta['test_metrics']['recall'])

Loaded Model: VotingClassifier
Model Name: Soft-Voting Ensemble
Optimized Decision Threshold: 0.38
Total Features: 12
Test ROC-AUC: 0.6698
Test Recall: 0.7305


## 3. Robust Inference Pipeline
Missing values in `ph`, `Sulfate`, and `Trihalomethanes` are handled using training medians alongside indicator flags.

In [3]:
BASE_FEATS = meta['base_features']
ALL_FEATS = meta['features']
MEDIANS = meta['train_medians']
THRESHOLD = meta['threshold']

def predict_water_quality(sample: dict) -> dict:
    row = {}
    for col in BASE_FEATS:
        val = sample.get(col)
        is_missing = val is None or (isinstance(val, float) and np.isnan(val))
        row[col] = MEDIANS[col] if is_missing else float(val)
        if f'{col}_was_missing' in ALL_FEATS:
            row[f'{col}_was_missing'] = int(is_missing)
    X = pd.DataFrame([row])[ALL_FEATS]
    prob = float(model.predict_proba(X)[0, 1])
    is_potable = bool(prob >= THRESHOLD)
    return {
        'probability_potable': round(prob, 4),
        'potable': is_potable,
        'status': 'SAFE' if is_potable else 'UNSAFE',
        'threshold': THRESHOLD
    }

sample = {'ph': 7.2, 'Hardness': 180.0, 'Solids': 18000.0, 'Chloramines': 6.5, 'Sulfate': 320.0, 'Conductivity': 410.0, 'Organic_carbon': 13.0, 'Trihalomethanes': 60.0, 'Turbidity': 3.5}
print('Sample Test Output:', predict_water_quality(sample))

Sample Test Output: {'probability_potable': 0.4058, 'potable': True, 'status': 'SAFE', 'threshold': 0.38}


## 4. Latency Profiling

In [4]:
N = 200
t0 = time.perf_counter()
for _ in range(N):
    _ = predict_water_quality(sample)
elapsed = time.perf_counter() - t0
print(f'Single Prediction Latency: {(elapsed / N) * 1000:.2f} ms')
print(f'Throughput: {N / elapsed:.0f} requests/sec')

Single Prediction Latency: 28.07 ms
Throughput: 36 requests/sec


## 5. Batch Inference on Sample Test Data

In [5]:
df_test = pd.read_csv('personC_app/sample_water_test.csv')
print('Loaded test batch:', df_test.shape)
preds = [predict_water_quality(r) for r in df_test.to_dict(orient='records')]
df_res = df_test.copy()
df_res['probability_potable'] = [p['probability_potable'] for p in preds]
df_res['potable'] = [p['potable'] for p in preds]
df_res[['ph', 'Hardness', 'probability_potable', 'potable']].head()

Loaded test batch: (20, 9)


,ph,Hardness,probability_potable,potable
0,8.14,264.27,0.5018,True
1,6.73,186.55,0.4733,True
2,NaN,200.03,0.5332,True
3,10.41,131.46,0.5137,True
4,6.52,171.91,0.3971,True


## 6. Streamlit Dashboard Architecture
The application is structured into 3 modular views:
1. **Predict:** Single sample inspection with gauge & SHAP waterfall, plus batch CSV upload & CSV download.
2. **Model Performance:** Cross-validation comparison, radar chart of test metrics, and baseline comparisons.
3. **Feature Insights:** Global SHAP summaries, permutation importance, and feature range distribution explorer.

Run the live dashboard locally:
```bash
streamlit run personC_app/app.py
```

In [6]:
print('Person C deliverables ready under personC_app/')
for item in sorted(os.listdir('personC_app')):
    print(' -', item)

Person C deliverables ready under personC_app/
 - Dockerfile
 - README.md
 - app.py
 - requirements.txt
 - sample_water_test.csv
